# 🧠 Omni-Modal Medical Diagnostic Framework
### GPU Training with CheXmultimodal Dataset (324 Chest X-rays + Clinical Text)

**Before starting:**
1. Set runtime to **GPU**: `Runtime → Change runtime type → T4 GPU`
2. Upload your `chexmultimodal` folder to Google Drive at: `MyDrive/btp_data/chexmultimodal/`

---

## 1️⃣ Setup Environment

In [ ]:
# Clone the repository
!git clone https://github.com/Ankit-blip737/Omni-Modal-Medical-Diagnostic-.git
%cd Omni-Modal-Medical-Diagnostic-
print('✅ Repository cloned')

In [ ]:
# Install dependencies (takes ~2 min)
!pip install -q torch torchvision torchaudio
!pip install -q transformers timm tqdm pyyaml scikit-learn matplotlib seaborn pandas tensorboard nibabel opencv-python
print('✅ All dependencies installed')

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    device = 'cuda'
else:
    print('⚠️ No GPU! Go to Runtime → Change runtime type → GPU')
    device = 'cpu'

## 2️⃣ Mount Google Drive & Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
import os

# === SET YOUR DATA PATH HERE ===
# Option 1: Data in Google Drive (recommended)
CHEX_DATA_PATH = '/content/drive/MyDrive/btp_data/chexmultimodal'

# Option 2: Upload directly to Colab (faster I/O but lost on disconnect)
# !mkdir -p /content/data
# # Upload via Colab file browser → drag and drop chexmultimodal folder
# CHEX_DATA_PATH = '/content/data/chexmultimodal'

# Verify dataset
if os.path.exists(CHEX_DATA_PATH):
    csv_files = [f for f in os.listdir(CHEX_DATA_PATH) if f.endswith('.csv')]
    img_dir = os.path.join(CHEX_DATA_PATH, 'new_patient_images_324')
    n_images = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    print(f'✅ Dataset found at {CHEX_DATA_PATH}')
    print(f'   CSV files: {csv_files}')
    print(f'   Images: {n_images}')
else:
    print(f'❌ Dataset NOT found at {CHEX_DATA_PATH}')
    print('Please upload your chexmultimodal folder to Google Drive at:')
    print('  MyDrive/btp_data/chexmultimodal/')
    print('  ├── new_indications_324.csv')
    print('  └── new_patient_images_324/')
    print('      ├── 0.jpg')
    print('      ├── 1.jpg')
    print('      └── ...')

In [ ]:
# Preview the dataset CSV
import pandas as pd

csv_path = os.path.join(CHEX_DATA_PATH, 'new_indications_324.csv')
df = pd.read_csv(csv_path)
print(f'Total samples: {len(df)}')
print(f'Columns: {list(df.columns)}')
print('\n--- First 5 rows ---')
df.head()

## 3️⃣ Build Model & Data Pipeline

In [ ]:
import sys
sys.path.insert(0, '.')

from src.models.omni_modal import OmniModalFramework
from src.data.chex_multimodal import get_chex_multimodal_dataloaders, CHEX_MULTIMODAL_LABELS
from src.training.trainer import OmniModalTrainer
from src.utils.logging import setup_logger, log_model_summary

# Build model (single modality for chest X-ray)
model = OmniModalFramework(
    num_modalities=1,          # Single chest X-ray
    num_classes=10,            # 10 pseudo-labels from indications
    text_model='pubmedbert',
    pretrained=True,
    freeze_image_backbone=True,
    freeze_text_layers=8,
    lora_rank=8,
    fusion_dim=768,
    projection_dim=256,
    num_fusion_stages=1,
    modality_dropout_p=0.0,
    multi_label=True,
)

log_model_summary(model)
model = model.to(device)
print(f'\n✅ Model loaded on {device}')

In [ ]:
# Create dataloaders with REAL data
data_config = {
    'root_dir': CHEX_DATA_PATH,
    'batch_size': 16,
    'num_workers': 2,
    'max_text_length': 128,
    'pin_memory': True,
}

loaders = get_chex_multimodal_dataloaders(data_config)
print(f'\n✅ Data loaded!')
print(f'   Train batches: {len(loaders["train"])}')
print(f'   Val batches:   {len(loaders["val"])}')
print(f'   Test batches:  {len(loaders["test"])}')

In [ ]:
# Visualize a sample batch
import matplotlib.pyplot as plt
import numpy as np

batch = next(iter(loaders['train']))
print(f'Batch images shape: {batch["images"].shape}')  # (B, 1, 1, 224, 224)
print(f'Batch labels shape: {batch["labels"].shape}')  # (B, 10)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < batch['images'].shape[0]:
        img = batch['images'][i, 0, 0].numpy()  # (224, 224)
        label_idxs = torch.where(batch['labels'][i] > 0)[0].tolist()
        label_names = [CHEX_MULTIMODAL_LABELS[j] for j in label_idxs]
        ax.imshow(img, cmap='gray')
        ax.set_title(', '.join(label_names), fontsize=9)
        ax.axis('off')
plt.suptitle('CheXmultimodal Training Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4️⃣ Phase 1: Contrastive Pre-training
Learns to align visual (X-ray) and textual (clinical indication) representations using InfoNCE loss.

In [ ]:
# Configure 3-phase curriculum training
training_config = {
    'mixed_precision': True,
    'gradient_clip_norm': 1.0,
    'log_every_n_steps': 5,
    'val_every_n_epochs': 1,
    'phase1': {
        'epochs': 30,
        'lr': 1e-4,
        'warmup_epochs': 3,
    },
    'phase2': {
        'epochs': 50,
        'lr': 5e-5,
        'weight_decay': 1e-4,
        'warmup_epochs': 5,
        'lambda_align': 0.1,
    },
    'phase3': {
        'epochs': 15,
        'lr': 1e-6,
        'weight_decay': 1e-5,
        'warmup_epochs': 2,
        'lambda_align': 0.05,
    },
}

trainer = OmniModalTrainer(
    model=model,
    train_loader=loaders['train'],
    val_loader=loaders['val'],
    config=training_config,
    device=device,
    log_dir='./logs',
    checkpoint_dir='./checkpoints',
)
print('✅ Trainer created')

In [ ]:
# Phase 1: Contrastive Pre-training (alignment learning)
print('=' * 60)
print('  PHASE 1: Contrastive Pre-training')
print('  Objective: Align visual & text representations')
print('=' * 60)
trainer.train_phase1()

## 5️⃣ Phase 2: Fusion Training
Trains the bidirectional cross-attention fusion engine with LoRA fine-tuning.

In [ ]:
# Phase 2: Fusion Training
print('=' * 60)
print('  PHASE 2: Fusion Training')
print('  Objective: Train cross-attention fusion + LoRA')
print('=' * 60)
trainer.train_phase2()

## 6️⃣ Phase 3: End-to-End Fine-tuning
Unfreezes all parameters for final full-model fine-tuning at low learning rate.

In [ ]:
# Phase 3: End-to-End Fine-tuning
print('=' * 60)
print('  PHASE 3: End-to-End Fine-tuning')
print('  Objective: Full parameter fine-tuning')
print('=' * 60)
trainer.train_phase3()

## 7️⃣ Evaluation on Test Set

In [ ]:
from src.evaluation.metrics import MetricTracker, format_metrics_table
from tqdm import tqdm

model.eval()
tracker = MetricTracker(class_names=CHEX_MULTIMODAL_LABELS)

with torch.no_grad():
    for batch in tqdm(loaders['test'], desc='Evaluating on test set'):
        images = batch['images'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        output = model(images, input_ids, attention_mask, compute_alignment_loss=False)
        probs = output['probabilities'].cpu().numpy()
        tracker.update(labels.numpy(), probs)

results = tracker.compute()
print('\n' + '=' * 60)
print('  TEST SET RESULTS')
print('=' * 60)
print(format_metrics_table(results))

## 8️⃣ Visualization

In [ ]:
from src.evaluation.visualization import plot_roc_curves, visualize_gate_values
import matplotlib.pyplot as plt

# ROC Curves per class
all_true = np.concatenate(tracker._all_true, axis=0)
all_prob = np.concatenate(tracker._all_prob, axis=0)

fig = plot_roc_curves(all_true, all_prob, CHEX_MULTIMODAL_LABELS)
plt.savefig('./results_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ ROC curves saved to results_roc_curves.png')

In [ ]:
# Gate Value Analysis — shows how the model weights visual vs text modality
model.eval()
gate_vs, gate_ts = [], []
sample_ids = []

with torch.no_grad():
    for i, batch in enumerate(loaders['test']):
        if i >= 5:
            break
        images = batch['images'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        output = model(images, input_ids, attention_mask)
        gate_vs.append(output['gate_values']['g_v'])
        gate_ts.append(output['gate_values']['g_t'])
        sample_ids.append(f'Batch {i}')

gate_v = np.array(gate_vs)
gate_t = np.array(gate_ts)

fig = visualize_gate_values(gate_v, gate_t, sample_ids=sample_ids)
plt.savefig('./results_gate_values.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Modality Weighting Analysis:')
print(f'   Avg Visual Gate (g_v): {gate_v.mean():.4f}')
print(f'   Avg Text Gate (g_t):   {gate_t.mean():.4f}')
print(f'   → Model relies more on: {"Vision" if gate_v.mean() > gate_t.mean() else "Text"}')

In [ ]:
# Confusion-style: Label distribution in predictions vs ground truth
import seaborn as sns

pred_labels = (all_prob > 0.5).astype(int)
true_counts = all_true.sum(axis=0)
pred_counts = pred_labels.sum(axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(CHEX_MULTIMODAL_LABELS))
width = 0.35
ax.bar(x - width/2, true_counts, width, label='Ground Truth', color='#2ecc71', alpha=0.8)
ax.bar(x + width/2, pred_counts, width, label='Predicted', color='#e74c3c', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(CHEX_MULTIMODAL_LABELS, rotation=45, ha='right')
ax.set_ylabel('Count')
ax.set_title('Label Distribution: Ground Truth vs Predictions')
ax.legend()
plt.tight_layout()
plt.savefig('./results_label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 9️⃣ Save Model & Download

In [ ]:
# Save final model
os.makedirs('./checkpoints', exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'num_modalities': 1,
    'num_classes': 10,
    'class_names': CHEX_MULTIMODAL_LABELS,
}, './checkpoints/omni_modal_chexmultimodal_final.pth')

print('✅ Model saved to ./checkpoints/omni_modal_chexmultimodal_final.pth')

# Also save to Google Drive for persistence
import shutil
drive_save_path = '/content/drive/MyDrive/btp_data/checkpoints/'
os.makedirs(drive_save_path, exist_ok=True)
shutil.copy('./checkpoints/omni_modal_chexmultimodal_final.pth', drive_save_path)
print(f'✅ Backup saved to Google Drive: {drive_save_path}')

In [ ]:
# Save all result plots to Drive too
results_drive = '/content/drive/MyDrive/btp_data/results/'
os.makedirs(results_drive, exist_ok=True)
for f in ['results_roc_curves.png', 'results_gate_values.png', 'results_label_distribution.png']:
    if os.path.exists(f):
        shutil.copy(f, results_drive)
print(f'✅ All results saved to {results_drive}')

## 🔟 TensorBoard (Optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./logs

---
## 📋 Quick Reference

| Phase | What it does | Epochs | Learning Rate |
|-------|-------------|--------|---------------|
| **Phase 1** | Contrastive alignment (InfoNCE) | 30 | 1e-4 |
| **Phase 2** | Fusion training (Cross-attention + LoRA) | 50 | 5e-5 |
| **Phase 3** | Full fine-tuning (all params) | 15 | 1e-6 |

**Total training time on T4 GPU: ~15-20 minutes**

### Architecture Summary
```
Chest X-ray → ConvNeXt-Base → 1024-dim → Project(768) ──┐
                                                          ├→ Cross-Attention Fusion → Classifier → 10 labels
Clinical Text → PubMedBERT + LoRA → 768-dim ─────────────┘
                                                    ↕
                                           CLIP-style Alignment
```